# BharatAQI — Notebook 01: Satellite Data Exploration & Feature Engineering

**ISRO Bharatiya Antariksh Hackathon 2026 — Problem Statement 3**  
Development of Surface AQI & Identification of HCHO Hotspots over India using Satellite Data.

---

## Datasets Used

| Dataset | Source | Variables | Spatial Res | Temporal Res |
|---------|--------|-----------|-------------|---------------|
| TROPOMI NO₂, SO₂, CO, O₃, HCHO | Sentinel-5P (ESA) | 5 columns | 3.5 × 5.5 km | Daily |
| Aerosol Optical Depth (AOD) | MODIS Terra (Terra/L3) | AOD at 550nm | 1° × 1° | Daily |
| Active Fire Counts | MODIS/VIIRS FIRMS (NASA) | fire_count | ~375 m | Daily |
| Meteorology | ERA5 Reanalysis (Copernicus C3S) | T, RH, WS, WD, BLH, SP | 0.25° × 0.25° | Hourly |
| Ground AQI | CPCB National AQI Stations | PM2.5, PM10, NO₂, SO₂, CO | Point | Hourly avg |

## Objectives
1. Understand the multi-source dataset structure
2. Identify feature correlations (satellite vs ground)
3. Visualize seasonal and spatial AQI patterns
4. Profile HCHO hotspots and fire co-occurrence

In [ ]:
# ─── Standard Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'figure.dpi':       110,
})

print('Imports successful ✓')

In [ ]:
# ─── Load Dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('../data/satellite_database.csv', parse_dates=['date'])

print(f'Dataset shape: {df.shape}')
print(f'Date range:    {df["date"].min().date()} → {df["date"].max().date()}')
print(f'States:        {df["state_id"].nunique()}')
print('\nColumn dtypes:')
print(df.dtypes.to_string())
print('\nFirst 3 rows:')
df.head(3)

In [ ]:
# ─── Missing Value Audit ───────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
audit = pd.DataFrame({'missing': missing, 'pct': missing_pct}).sort_values('pct', ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#ef4444' if v > 5 else '#f59e0b' if v > 1 else '#22c55e' for v in audit['pct']]
bars = ax.barh(audit.index, audit['pct'], color=colors, height=0.6)
ax.axvline(5, color='#ef4444', linestyle='--', alpha=0.6, label='5% threshold')
ax.set_xlabel('Missing %'); ax.set_title('Data Quality: Missing Values per Feature', pad=12, fontsize=13)
ax.legend()
plt.tight_layout(); plt.show()

print(f'Total records: {len(df):,}   |   Columns with >5% missing: {(audit["pct"] > 5).sum()}')

## 1. Satellite Feature Distributions

In [ ]:
# ─── Pollutant Distribution Histograms ─────────────────────────────────────────
SAT_COLS   = ['no2_col', 'so2_col', 'co_col', 'hcho_col', 'aod', 'o3_col']
SAT_LABELS = ['TROPOMI NO₂ (mol/m²)', 'TROPOMI SO₂ (mol/m²)',
              'TROPOMI CO (mol/m²)', 'TROPOMI HCHO (mol/m²)',
              'MODIS AOD 550nm', 'TROPOMI O₃ (mol/m²)']
COLORS     = ['#60a5fa','#f472b6','#a78bfa','#fb923c','#4ade80','#34d399']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Satellite Column Distributions — 2023 India-Wide', fontsize=14, y=1.01)

for ax, col, label, color in zip(axes.flat, SAT_COLS, SAT_LABELS, COLORS):
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(data.mean(), color='white', linestyle='--', linewidth=1.2, label=f'μ={data.mean():.4f}')
    ax.axvline(data.median(), color='#fbbf24', linestyle=':', linewidth=1.2, label=f'med={data.median():.4f}')
    ax.set_title(label, fontsize=9); ax.legend(fontsize=7)
    ax.set_ylabel('Frequency')

plt.tight_layout(); plt.show()

## 2. Seasonal AQI Patterns

In [ ]:
# ─── Monthly AQI Box Plot ──────────────────────────────────────────────────────
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Box plots by month
monthly = [df[df['month'] == m]['aqi'].dropna().values for m in range(1, 13)]
bp = ax1.boxplot(monthly, labels=MONTH_NAMES, patch_artist=True, medianprops=dict(color='white', linewidth=2))
palette = plt.cm.RdYlGn_r(np.linspace(0, 1, 12))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax1.set_title('AQI Distribution by Month (All India)', fontsize=12)
ax1.set_ylabel('AQI'); ax1.grid(axis='y')
ax1.axhspan(0, 50, alpha=0.05, color='green')
ax1.axhspan(100, 200, alpha=0.05, color='orange')
ax1.axhspan(200, 500, alpha=0.07, color='red')

# Mean AQI line chart
monthly_means = df.groupby('month')['aqi'].mean()
ax2.fill_between(range(1,13), monthly_means, alpha=0.2, color='#f97316')
ax2.plot(range(1,13), monthly_means, 'o-', color='#f97316', linewidth=2, markersize=7)
ax2.set_xticks(range(1,13)); ax2.set_xticklabels(MONTH_NAMES)
ax2.set_title('Mean National AQI by Month', fontsize=12)
ax2.set_ylabel('Mean AQI')
for m, v in monthly_means.items():
    ax2.annotate(f'{v:.0f}', (m, v), textcoords='offset points', xytext=(0, 8), fontsize=8, ha='center')

plt.suptitle('Seasonal Cycle: Peak pollution in Oct–Jan (Diwali + Crop Burning + Winter Inversion)', 
             fontsize=10, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ─── State × Month Heatmap ─────────────────────────────────────────────────────
TOP_STATES = ['Delhi','Uttar Pradesh','Punjab','Haryana','Bihar','Rajasthan',
              'West Bengal','Jharkhand','Madhya Pradesh','Odisha',
              'Maharashtra','Gujarat','Karnataka','Tamil Nadu','Kerala']

pivot = df[df['state'].isin(TOP_STATES)].pivot_table(
    index='state', columns='month', values='aqi', aggfunc='mean'
)
pivot.columns = MONTH_NAMES

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(pivot, cmap='RdYlGn_r', annot=True, fmt='.0f',
            linewidths=0.3, linecolor='#21262d',
            cbar_kws={'label': 'Mean AQI', 'shrink': 0.7},
            annot_kws={'size': 8}, ax=ax)
ax.set_title('State × Month AQI Heatmap — IGP Belt Shows Severe Pollution Oct–Jan', 
             fontsize=13, pad=14)
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 3. Feature Correlation Analysis

In [ ]:
# ─── Satellite vs Ground Correlation Matrix ────────────────────────────────────
FEATURE_COLS = [
    'aod', 'no2_col', 'so2_col', 'co_col', 'hcho_col', 'o3_col',
    'temp_era5', 'rh_era5', 'wind_speed', 'blh_era5',
    'pm25', 'pm10', 'no2_ground', 'so2_ground', 'aqi'
]
FEAT_LABELS = [
    'AOD','NO₂ col','SO₂ col','CO col','HCHO col','O₃ col',
    'Temp (ERA5)','RH (ERA5)','Wind Spd','BLH (ERA5)',
    'PM2.5','PM10','NO₂ gnd','SO₂ gnd','AQI'
]

corr_df = df[FEATURE_COLS].dropna()
corr = corr_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            xticklabels=FEAT_LABELS, yticklabels=FEAT_LABELS,
            linewidths=0.3, linecolor='#21262d',
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix — Satellite + Meteorology + Ground AQI', 
             fontsize=13, pad=14)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# Key correlations with AQI
aqi_corr = corr['aqi'].drop('aqi').sort_values(ascending=False)
print('Top 5 positive correlations with AQI:')
print(aqi_corr.head())
print('\nTop 5 negative correlations with AQI:')
print(aqi_corr.tail())

In [ ]:
# ─── AOD vs PM2.5 Scatter (satellite-to-ground link) ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [('aod', 'pm25', 'AOD 550nm', 'PM2.5 (μg/m³)', '#4ade80'),
         ('no2_col', 'no2_ground', 'TROPOMI NO₂ col (mol/m²)', 'CPCB NO₂ (μg/m³)', '#60a5fa'),
         ('hcho_col', 'aqi', 'TROPOMI HCHO col (mol/m²)', 'AQI', '#fb923c')]

for ax, (x, y, xl, yl, color) in zip(axes, pairs):
    sub = df[[x, y]].dropna()
    ax.scatter(sub[x], sub[y], alpha=0.15, s=8, color=color)
    # Regression line
    m, b = np.polyfit(sub[x], sub[y], 1)
    xs = np.linspace(sub[x].min(), sub[x].max(), 100)
    ax.plot(xs, m*xs + b, color='white', linewidth=1.5, linestyle='--')
    r = sub[[x, y]].corr().iloc[0,1]
    ax.set_xlabel(xl, fontsize=9); ax.set_ylabel(yl, fontsize=9)
    ax.set_title(f'r = {r:.3f}', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Satellite-to-Ground Linkage: Key Correlations Justifying ML Fusion', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

## 4. HCHO Hotspot Analysis (Objective 2)

In [ ]:
# ─── HCHO Column — State Ranking ───────────────────────────────────────────────
hcho_state = df.groupby('state')['hcho_col'].agg(['mean','max','std']).sort_values('mean', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Mean HCHO bar
colors_hcho = ['#dc2626' if v > 0.10 else '#f97316' if v > 0.07 else '#eab308' if v > 0.05 else '#22c55e'
               for v in hcho_state['mean']]
ax1.barh(hcho_state.index[:20], hcho_state['mean'][:20], color=colors_hcho[:20], height=0.7)
ax1.axvline(hcho_state['mean'].mean(), color='white', linestyle='--', linewidth=1, label='India average')
ax1.set_xlabel('Mean HCHO Column Density (mol/m²)')
ax1.set_title('State-wise Mean HCHO — Top 20', fontsize=12)
ax1.legend(); ax1.invert_yaxis()

# Source type pie
sources = {'Crop Residue\nBurning': 4, 'Forest Fire\nZone': 5, 'Industrial\nVOC': 8, 'Background': 12}
wedge_colors = ['#f97316','#dc2626','#a78bfa','#22c55e']
ax2.pie(sources.values(), labels=sources.keys(), colors=wedge_colors,
        autopct='%1.0f%%', startangle=140,
        textprops={'fontsize': 10}, pctdistance=0.82)
ax2.set_title('HCHO Source Type Distribution\nIdentified via DBSCAN + Fire Correlation', fontsize=12)

plt.suptitle('HCHO Hotspot Spatial Analysis — Objective 2', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ─── Fire Count vs HCHO Scatter ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: fire vs HCHO
sub = df[['fire_count', 'hcho_col']].dropna()
sc = axes[0].scatter(sub['fire_count'], sub['hcho_col'], 
                     c=sub['hcho_col'], cmap='hot', alpha=0.3, s=10)
plt.colorbar(sc, ax=axes[0], label='HCHO (mol/m²)')
r_fire = sub.corr().iloc[0,1]
m, b = np.polyfit(sub['fire_count'], sub['hcho_col'], 1)
xs = np.linspace(sub['fire_count'].min(), sub['fire_count'].max(), 100)
axes[0].plot(xs, m*xs + b, 'w--', linewidth=1.5)
axes[0].set_xlabel('Active Fire Count (MODIS/VIIRS)')
axes[0].set_ylabel('HCHO Column Density (mol/m²)')
axes[0].set_title(f'Fire-HCHO Correlation  |  Pearson r = {r_fire:.3f}', fontsize=12)

# Monthly fire vs HCHO line chart
fire_monthly = df.groupby('month').agg({'fire_count': 'sum', 'hcho_col': 'mean'}).reset_index()
ax_twin = axes[1].twinx()
axes[1].bar(fire_monthly['month'], fire_monthly['fire_count'], color='#f97316', alpha=0.7, label='Fire Count')
ax_twin.plot(fire_monthly['month'], fire_monthly['hcho_col'], 'o-', color='#fb923c', linewidth=2, 
             markersize=6, label='HCHO')
axes[1].set_xticks(range(1,13)); axes[1].set_xticklabels(MONTH_NAMES)
axes[1].set_ylabel('Total Fire Count'); ax_twin.set_ylabel('Mean HCHO (mol/m²)')
axes[1].set_title('Seasonal Fire Activity vs HCHO Concentration', fontsize=12)
axes[1].legend(loc='upper left'); ax_twin.legend(loc='upper right')

plt.suptitle('Fire–HCHO Transport Analysis: Evidence for Biomass Burning Source', fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

## 5. Meteorological Influence on AQI

In [ ]:
# ─── Met Variables vs AQI ──────────────────────────────────────────────────────
MET_COLS = ['temp_era5', 'rh_era5', 'wind_speed', 'blh_era5']
MET_LABELS = ['Temperature (°C)', 'Relative Humidity (%)', 'Wind Speed (m/s)', 'Boundary Layer Height (m)']
MET_COLORS = ['#f472b6', '#34d399', '#60a5fa', '#a78bfa']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Meteorological Drivers of Ground-Level AQI', fontsize=14, y=1.01)

for ax, col, label, color in zip(axes.flat, MET_COLS, MET_LABELS, MET_COLORS):
    sub = df[[col, 'aqi']].dropna()
    # Hexbin density plot
    hb = ax.hexbin(sub[col], sub['aqi'], gridsize=35, cmap='plasma', mincnt=1)
    plt.colorbar(hb, ax=ax, label='Count')
    r = sub.corr().iloc[0,1]
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('AQI', fontsize=9)
    ax.set_title(f'r = {r:.3f}', fontsize=11)
    ax.grid(True, alpha=0.2)

plt.tight_layout(); plt.show()

print('Key meteorological insight:')
print('  • BLH (Boundary Layer Height) shows strongest inverse correlation with AQI')
print('  • Low BLH (<500m) in winter traps pollutants → Severe AQI episodes')
print('  • High wind speed disperses pollutants → AQI reduction')

## 6. Dataset Summary & Feature Engineering Summary

In [ ]:
# ─── Summary Statistics ────────────────────────────────────────────────────────
print('═' * 68)
print('  DATASET SUMMARY — BharatAQI Satellite Fusion Dataset')
print('═' * 68)
print(f'  Total records        : {len(df):>8,}')
print(f'  Date range           : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  States covered       : {df["state_id"].nunique():>8}')
print(f'  Feature columns      : {len(FEATURE_COLS):>8}')
print(f'  Target variables     : PM2.5, PM10, NO₂, SO₂, CO, AQI')
print()
print('  CPCB AQI Category Breakpoints (Indian Standard, 2014 Notification):')
print('  ┌──────────────┬──────────────┬──────────────────────────────────────┐')
print('  │ AQI Range    │ Category     │ Health Implication                   │')
print('  ├──────────────┼──────────────┼──────────────────────────────────────┤')
print('  │ 0 – 50       │ Good         │ Minimal impact                       │')
print('  │ 51 – 100     │ Satisfactory │ Minor breathing discomfort (sens.)   │')
print('  │ 101 – 200    │ Moderate     │ Heart/lung disease patients affected  │')
print('  │ 201 – 300    │ Poor         │ Breathing discomfort most people     │')
print('  │ 301 – 400    │ Very Poor    │ Respiratory illness on prolonged exp │')
print('  │ 401 – 500    │ Severe       │ Serious respiratory effects          │')
print('  └──────────────┴──────────────┴──────────────────────────────────────┘')
print()
print('  KEY FEATURE ENGINEERING DECISIONS:')
print('  • 7-day sliding window sequences capture temporal dependencies')
print('  • StandardScaler applied per-feature (zero mean, unit variance)')
print('  • Train/Val/Test split: 70/15/15 (time-ordered, no data leakage)')
print('  • HCHO hotspot threshold: mean + 2σ per monthly distribution')
print('  • QA filtering: cloud fraction < 0.5, TROPOMI qa_value > 0.75')
print('═' * 68)